In [ ]:
"""
Probabilistic coral age model (PyMC)
-----------------------------------
Data inputs:
  - /mnt/data/lab_results.xlsx
  - /mnt/data/modeled_d18o.csv

What this model does:
  1) Builds a GLOBAL ring-index "event spine" from X-ray band picks:
        R_j = R0 + cumsum(age1_j)
     and uses age2 (sparse) to anchor R to your manual cumulative numbering.

  2) Maps ring index to time via the 2007-04 collection anchor:
        months_before_end ≈ 12 * (R_end - R_j)
     (with an estimated band-timing uncertainty sigma_band)

  3) Fits a per-path MONOTONE time–distance function (piecewise linear, knot-based)
     so every sample i gets a latent time (months_before_end_i), and this is
     constrained by the band events that occur on that path.

  4) Adds optional climate δ18O alignment for the recent portion (e.g., last 20y),
     plus a flexible seasonal “shape” constraint using harmonics (no need to
     compute calendar months inside the graph).

Outputs:
  - Posterior samples of months_before_end for every sample (=> calendar ages)
  - Posterior uncertainty for event times, growth shapes, climate fit parameters
"""

import numpy as np
import pandas as pd

import pymc as pm
import pytensor
import pytensor.tensor as pt

# -------------------------
# 0) Load data
# -------------------------
LAB_PATH = "../../2025-12-20_AgeModel-NewXrayPatch/age_model/lab_results.xlsx"
CLIM_PATH = "../../2026-01-15_Climate-Model/analysis/modeled_d18o.csv"

lab = pd.read_excel(LAB_PATH)
clim = pd.read_csv(CLIM_PATH)

# Ensure required columns exist
required_lab_cols = {"index", "age1", "age2", "path_name", "path_num", "d18o"}
missing = required_lab_cols - set(lab.columns)
if missing:
    raise ValueError(f"lab_results.xlsx missing columns: {missing}")

# Collection anchor (hard constraint): end date is April 2007
END_DATE = np.datetime64("2007-04-15")  # choose mid-month unless you know exact day

# Normalize coral d18O (mean-subtracted as you described)
lab = lab.sort_values("index").reset_index(drop=True)
lab["d18o_norm"] = lab["d18o"] - np.nanmean(lab["d18o"].values)

# Climate series: parse dates, confirm mean-subtracted
clim["date"] = pd.to_datetime(clim["date"])
clim = clim.sort_values("date").reset_index(drop=True)
clim["d18o"] = clim["d18o"].astype(float)

# Convert climate time to "months before END_DATE"
# Use 30.4375 days/month as a stable conversion for continuous-time inference
DAYS_PER_MONTH = 30.4375
clim_days_before = (END_DATE.astype("datetime64[D]") - clim["date"].values.astype("datetime64[D]")).astype(int)
clim_months_before = clim_days_before / DAYS_PER_MONTH
clim_x = clim["d18o"].values.astype(float)

# Keep only climate points that are <= end date (months_before >= 0)
mask_clim = clim_months_before >= 0
clim_months_before = clim_months_before[mask_clim]
clim_x = clim_x[mask_clim]

# -------------------------
# 1) Build X-ray band "events" in master order
# -------------------------
# Band start events: rows where age1 is not NaN (your band start marker)
events = lab.loc[lab["age1"].notna(), ["index", "path_name", "path_num", "age1", "age2"]].copy()
events = events.sort_values("index").reset_index(drop=True)

# Observed deltas between ring indices (exact, from age1 entries)
deltas = events["age1"].astype(float).values  # first delta is not used as "difference from previous"
# We interpret the first event as baseline; subsequent events jump by deltas[j] relative to previous.
# So build a relative ring index sequence R_rel with R_rel[0]=0, R_rel[j]=sum_{k=1..j} deltas[k]
R_rel = np.zeros(len(events), dtype=float)
if len(events) > 1:
    R_rel[1:] = np.cumsum(deltas[1:])

events["R_rel"] = R_rel

# Sparse age2 observations (your manual cumulative helper); can be fractional
age2_obs = events["age2"].values.astype(float)
age2_mask = np.isfinite(age2_obs)
age2_obs_idx = np.where(age2_mask)[0]
age2_obs_val = age2_obs[age2_mask]

# -------------------------
# 2) Prepare per-sample path geometry (monotone time–distance)
# -------------------------
# We will model months_before_end as a monotone INCREASING function of "distance from rim"
# because older samples are further from rim and have larger months_before_end.
# Given your sampling order center -> rim (younger later), path_num increases toward rim.
# Define rev distance r = max(path_num in path) - path_num so r=0 at rim, increasing inward.
lab["path_name"] = lab["path_name"].astype(str)
paths = sorted(lab["path_name"].unique().tolist())

# Encode paths as integer indices
path_to_idx = {p: k for k, p in enumerate(paths)}
lab["path_idx"] = lab["path_name"].map(path_to_idx).astype(int)

# Compute r (distance from rim) within each path
lab["r"] = np.nan
for p in paths:
    m = lab["path_name"] == p
    max_pn = np.nanmax(lab.loc[m, "path_num"].values.astype(float))
    lab.loc[m, "r"] = max_pn - lab.loc[m, "path_num"].values.astype(float)

r_all = lab["r"].values.astype(float)
path_idx_all = lab["path_idx"].values.astype(int)

# For each path, choose knot positions in r-space
# Keep knot count modest for NUTS stability.
KNOTS = 12  # adjust (e.g., 10–20) depending on how wiggly you want the warps
knot_r = {}
for p in paths:
    m = lab["path_name"] == p
    r_p = lab.loc[m, "r"].values.astype(float)
    r_min, r_max = float(np.min(r_p)), float(np.max(r_p))
    # Evenly spaced knots inclusive
    knot_r[p] = np.linspace(r_min, r_max, KNOTS).astype("float64")

# Map each sample to its path-specific knot grid
# We'll do interpolation inside the PyMC graph, so we need:
#   - r_i for each sample
#   - knot_r_p (KNOTS) for each path p
# and parameters for knot values (months_before at knots).

# Also need to link EVENT rows to sample rows:
# events reference (path_name, path_num). Find matching sample index in lab.
# (Assumes unique path_num within each path.)
lab_key = lab.set_index(["path_name", "path_num"])
event_sample_idx = []
for _, row in events.iterrows():
    key = (row["path_name"], row["path_num"])
    if key not in lab_key.index:
        raise ValueError(f"Event sample not found in lab table for key={key}")
    # Find integer row index in lab
    # Using boolean mask is robust to duplicate indices; but should be unique.
    i = lab.index[(lab["path_name"] == row["path_name"]) & (lab["path_num"] == row["path_num"])][0]
    event_sample_idx.append(int(i))
event_sample_idx = np.array(event_sample_idx, dtype=int)

# -------------------------
# 3) Helper: linear interpolation in PyTensor
# -------------------------
def linear_interp(x, xp, fp):
    """
    Vectorized 1D linear interpolation for PyTensor.

    x:  (N,) points
    xp: (K,) increasing grid
    fp: (K,) values on grid

    Returns:
      y: (N,)
    """
    # searchsorted returns insertion index in [0..K]
    idx = pt.extra_ops.searchsorted(xp, x, side="left")

    # clip to valid segment indices [1..K-1]
    idx = pt.clip(idx, 1, xp.shape[0] - 1)

    x0 = xp[idx - 1]
    x1 = xp[idx]
    y0 = fp[idx - 1]
    y1 = fp[idx]

    w = (x - x0) / (x1 - x0)
    return y0 + w * (y1 - y0)

# -------------------------
# 4) Climate interpolation grid (months_before -> climate d18O)
# -------------------------
# Build a monotone grid in months_before for stable interpolation
order = np.argsort(clim_months_before)
clim_mb_grid = clim_months_before[order].astype("float64")
clim_x_grid = clim_x[order].astype("float64")

# Trim duplicates in grid (rare but safe)
# If there are duplicates, average them.
dfc = pd.DataFrame({"mb": clim_mb_grid, "x": clim_x_grid}).groupby("mb", as_index=False)["x"].mean()
clim_mb_grid = dfc["mb"].values.astype("float64")
clim_x_grid = dfc["x"].values.astype("float64")

# -------------------------
# 5) Build the PyMC model
# -------------------------
y_obs = lab["d18o_norm"].values.astype("float64")
N = len(lab)
P = len(paths)
J = len(events)

# Overlap gating for climate anchoring (e.g., last 20 years)
OVERLAP_YEARS = 20
overlap_months = 12 * OVERLAP_YEARS
overlap_mask = (r_all >= 0) & (np.isfinite(y_obs))  # basic sanity; climate gating is in-graph below

with pm.Model() as model:

    # -------------------------
    # 5A) Ring index spine (events)
    # -------------------------
    # Global offset for ring index scale (aligns R_rel to your age2 numbering)
    R0 = pm.Normal("R0", mu=0.0, sigma=50.0)

    # Event ring indices (deterministic from R0 and observed deltas recursion)
    R_event = pm.Deterministic("R_event", R0 + pm.math.constant(R_rel))

    # Age2 anchoring (sparse)
    sigma_age2 = pm.HalfNormal("sigma_age2", sigma=1.0)
    pm.Normal("age2_like", mu=R_event[age2_obs_idx], sigma=sigma_age2, observed=age2_obs_val)

    # Ring index at collection/end date (must be >= all event ring indices)
    # Enforce with a positive slack via softplus
    slack = pm.HalfNormal("Rend_slack", sigma=5.0)
    R_end = pm.Deterministic("R_end", pm.math.max(R_event) + slack)

    # Band timing uncertainty (months)
    sigma_band = pm.HalfNormal("sigma_band", sigma=2.0)

    # Expected months_before_end for each EVENT given ring index
    mb_event_expected = pm.Deterministic("mb_event_expected", 12.0 * (R_end - R_event))

    # -------------------------
    # 5B) Per-path monotone time–distance functions (knot-based, piecewise linear)
    # -------------------------
    # For each path p:
    #   mb_p(r) = base_p + cumsum(exp(deltas_p)) interpolated over knot_r[p]
    #
    # base_p is months_before_end at rim (r=0). It must be >=0.
    base_p = pm.HalfNormal("base_p", sigma=60.0, shape=P)  # allow up to ~5 years typical; widen if needed

    # Positive increments between knots; exp ensures monotone increasing in r
    # Use moderately informative scale to avoid crazy warps.
    knot_step_raw = pm.Normal("knot_step_raw", mu=-1.0, sigma=1.0, shape=(P, KNOTS - 1))
    knot_steps = pm.Deterministic("knot_steps", pm.math.exp(knot_step_raw))  # >0

    # Build knot values for each path: mb at knot k
    # mb_knots[p,0] = base_p
    # mb_knots[p,k] = base_p + sum_{m<k} knot_steps[p,m]
    mb_knots = pm.Deterministic(
        "mb_knots",
        pt.concatenate(
            [base_p[:, None], base_p[:, None] + pt.cumsum(knot_steps, axis=1)],
            axis=1
        )
    )

    # Interpolate each sample's months_before_end based on its path and r
    # We do this path-by-path and stitch results into a length-N vector.
    mb_samples_parts = []
    for p_i, p in enumerate(paths):
        m = (path_idx_all == p_i)
        r_p = r_all[m].astype("float64")
        # Make these constants in the graph:
        r_p_t = pm.math.constant(r_p)
        knot_r_p_t = pm.math.constant(knot_r[p])
        mb_p_t = linear_interp(r_p_t, knot_r_p_t, mb_knots[p_i])
        mb_samples_parts.append(mb_p_t)

    # Concatenate in the same order as lab rows
    # We need to place each path-specific vector back into the full N vector.
    mb_samples = pt.zeros((N,), dtype="float64")
    cursor = 0
    for p_i, p in enumerate(paths):
        m = (path_idx_all == p_i)
        n_p = int(np.sum(m))
        mb_samples = pt.set_subtensor(mb_samples[np.where(m)[0]], mb_samples_parts[cursor])
        cursor += 1

    mb_samples = pm.Deterministic("mb_samples", mb_samples)

    # Hard anchor constraint: all samples must be <= end date => months_before_end >= 0
    # Our construction guarantees nonnegative because base_p >=0 and steps >0, but add a tiny jitter guard:
    pm.Potential("nonneg_guard", pt.switch(pt.all(mb_samples >= -1e-6), 0.0, -np.inf))

    # -------------------------
    # 5C) Link EVENT constraints to per-path time warps
    # -------------------------
    # The sample at each event should line up with the event-time expectation from the ring spine.
    pm.Normal(
        "event_time_like",
        mu=mb_event_expected,
        sigma=sigma_band,
        observed=mb_samples[event_sample_idx]
    )

    # -------------------------
    # 5D) Climate + seasonal likelihood for coral d18O
    # -------------------------
    # Linear climate scaling + optional recent anchoring
    a0 = pm.Normal("a0", mu=0.0, sigma=0.5)      # small offset (should be near 0 if mean-subtracted)
    b0 = pm.Normal("b0", mu=1.0, sigma=0.5)      # climate amplitude scaling

    # Seasonal "shape" constraint using first 2 harmonics (flexible peaks/flatness)
    # seasonal(mb) = s1*sin(2π mb/12) + c1*cos(2π mb/12) + s2*sin(4π mb/12) + c2*cos(4π mb/12)
    s1 = pm.Normal("s1", mu=0.0, sigma=0.5)
    c1 = pm.Normal("c1", mu=0.0, sigma=0.5)
    s2 = pm.Normal("s2", mu=0.0, sigma=0.5)
    c2 = pm.Normal("c2", mu=0.0, sigma=0.5)

    omega1 = 2.0 * np.pi / 12.0
    omega2 = 2.0 * omega1

    seasonal = (
        s1 * pt.sin(omega1 * mb_samples) + c1 * pt.cos(omega1 * mb_samples) +
        s2 * pt.sin(omega2 * mb_samples) + c2 * pt.cos(omega2 * mb_samples)
    )

    # Interpolate climate x at inferred months_before_end
    clim_mb_t = pm.math.constant(clim_mb_grid)
    clim_x_t = pm.math.constant(clim_x_grid)
    x_at_t = linear_interp(mb_samples, clim_mb_t, clim_x_t)

    # Gate climate anchoring to recent portion (last 20y by default)
    # This keeps older portions from being forced to match a forward model that may be weak there.
    use_climate = pm.math.le(mb_samples, overlap_months)
    x_term = pm.math.switch(use_climate, b0 * x_at_t, 0.0)

    # Observation noise (StudentT is robust; switch to Normal if you prefer)
    sigma_y = pm.HalfNormal("sigma_y", sigma=0.5)
    nu = pm.Exponential("nu", lam=1/10) + 2.0  # df > 2

    mu_y = a0 + x_term + seasonal

    pm.StudentT("d18o_like", nu=nu, mu=mu_y, sigma=sigma_y, observed=y_obs)

    # -------------------------
    # 5E) Sampling
    # -------------------------
    # Suggested run settings; start small, then scale up.
    # If you get divergences, tighten knot priors or reduce KNOTS.
    idata = pm.sample(
        draws=1500,
        tune=1500,
        chains=4,
        target_accept=0.9,
        init="jitter+adapt_diag",
        random_seed=42,
    )

# -------------------------
# 6) Post-processing helpers (calendar ages)
# -------------------------
# Convert posterior mb_samples (months_before_end) to approximate calendar dates outside the graph:
# date_i = END_DATE - mb_i * DAYS_PER_MONTH (days)
import arviz as az

mb_post = idata.posterior["mb_samples"]  # dims: chain, draw, sample
mb_mean = mb_post.mean(dim=("chain", "draw")).values
mb_hdi = az.hdi(mb_post, hdi_prob=0.95).to_array().values  # shape (2, sample)

# Approximate dates (days)
days_mean = (mb_mean * DAYS_PER_MONTH).astype(int)
date_mean = END_DATE.astype("datetime64[D]") - days_mean.astype("timedelta64[D]")

days_lo = (mb_hdi[0] * DAYS_PER_MONTH).astype(int)
days_hi = (mb_hdi[1] * DAYS_PER_MONTH).astype(int)
date_lo = END_DATE.astype("datetime64[D]") - days_hi.astype("timedelta64[D]")  # older => larger days
date_hi = END_DATE.astype("datetime64[D]") - days_lo.astype("timedelta64[D]")  # younger => smaller days

out = lab.copy()
out["age_date_mean"] = date_mean
out["age_date_95lo"] = date_lo
out["age_date_95hi"] = date_hi
out.to_csv("/mnt/data/coral_age_posterior_summary.csv", index=False)

print("Wrote posterior age summary to: /mnt/data/coral_age_posterior_summary.csv")
print("Key posterior variables: R_end, sigma_band, base_p, mb_samples, climate/seasonal params")
